# Fine-Tuning, LoRA & QLoRA — Google Colab Notebook

## Practical class notebook for intermediate students

This notebook explains and demonstrates **parameter-efficient fine-tuning** using:

- Full fine-tuning concept
- LoRA
- QLoRA
- Hugging Face Transformers
- PEFT
- bitsandbytes 4-bit quantization
- A small instruction dataset
- Adapter training
- Inference after fine-tuning

> This notebook is designed for teaching. It uses a small dataset so students can understand the complete workflow without waiting for long training.


## Learning Outcomes

By the end of this notebook, students should be able to:

1. Explain what fine-tuning is.
2. Explain why full fine-tuning is expensive.
3. Understand LoRA as a parameter-efficient fine-tuning method.
4. Understand QLoRA as quantized LoRA.
5. Prepare a small instruction dataset.
6. Load a base model.
7. Add LoRA adapters.
8. Train only adapter parameters.
9. Run inference after adapter training.
10. Understand common hyperparameters such as rank, alpha, dropout, learning rate, and batch size.


# 1. Theory Recap

## What is Fine-Tuning?

Fine-tuning means taking a **pre-trained model** and training it further on a **specific dataset**.

Example:

A general language model may already know English, programming, science, and general facts.  
If we train it on customer-support conversations, it can become better at answering support-related questions.

## Why not always use full fine-tuning?

Full fine-tuning updates **all model parameters**.

For large language models, this requires:

- Large GPU memory
- Long training time
- High cost
- Large model checkpoints
- More engineering effort

This is why **parameter-efficient fine-tuning** is useful.


# 2. LoRA Explanation

## What is LoRA?

**LoRA** stands for **Low-Rank Adaptation**.

Instead of updating all original weights of the model, LoRA freezes the base model and trains small additional matrices.

Simple idea:

```text
Original weight matrix W stays frozen.
LoRA adds small trainable matrices A and B.
Only A and B are updated during training.
```

Mathematical idea:

```text
W' = W + BA
```

Where:

- `W` = original frozen weight matrix
- `A` and `B` = small trainable low-rank matrices
- `r` = rank, controls adapter size


# 3. QLoRA Explanation

## What is QLoRA?

**QLoRA** means **Quantized LoRA**.

It combines:

```text
4-bit quantized base model + LoRA adapters
```

The base model is loaded in low memory using 4-bit quantization, while LoRA adapters are trained normally.

This allows larger models to be fine-tuned on limited GPU memory.

## Key idea

```text
LoRA = Freeze base model + train adapter weights
QLoRA = Quantize base model to 4-bit + train adapter weights
```


# 4. Colab Runtime Setup

Before running this notebook:

1. Go to **Runtime**
2. Click **Change runtime type**
3. Select **GPU**
4. Save

Then run the next cell to check the GPU.


In [19]:
!nvidia-smi

Sun Aug  9 08:16:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             17W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# 5. Install Required Libraries

We need the following libraries:

- `transformers` for loading language models
- `datasets` for dataset handling
- `peft` for LoRA adapters
- `bitsandbytes` for QLoRA 4-bit quantization
- `accelerate` for efficient model loading
- `sentencepiece` and `protobuf` for tokenizer support


In [20]:
%pip install -q -U \
  transformers \
  datasets \
  peft \
  bitsandbytes \
  accelerate \
  sentencepiece \
  protobuf

# 6. Import Libraries

In [21]:
import os
import torch
import pandas as pd

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. QLoRA training may not work properly on CPU.")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


# 7. Configuration

For classroom demonstration, we use a small instruction model.

You can change the base model later, but start small for faster testing.

## Modes

```python
USE_QLORA = True
```

This loads the base model in 4-bit and trains LoRA adapters.

```python
USE_QLORA = False
```

This loads the model normally and trains LoRA adapters without 4-bit quantization.


In [22]:
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

USE_QLORA = True

OUTPUT_DIR = "/content/lora_qlora_adapter"

MAX_LENGTH = 512

print("Base model:", BASE_MODEL)
print("Using QLoRA:", USE_QLORA)
print("Output directory:", OUTPUT_DIR)

Base model: Qwen/Qwen2.5-0.5B-Instruct
Using QLoRA: True
Output directory: /content/lora_qlora_adapter
Base model: Qwen/Qwen2.5-0.5B-Instruct
Using QLoRA: True
Output directory: /content/lora_qlora_adapter


# 8. Create a Small Instruction Dataset

Fine-tuning needs training examples.

Each example should contain:

```text
instruction/question → expected response
```

For this class, we create a small dataset about Generative AI, RAG, LoRA, QLoRA, and prompt engineering.

In real projects, your dataset should be much larger and domain-specific.


In [23]:
training_data = [
    {
        "instruction": "What is Generative AI?",
        "response": "Generative AI is a type of artificial intelligence that can create new content such as text, images, audio, video, and code."
    },
    {
        "instruction": "Explain RAG in simple words.",
        "response": "RAG stands for Retrieval-Augmented Generation. It retrieves relevant information from documents and gives that context to a language model so it can answer more accurately."
    },
    {
        "instruction": "What is fine-tuning?",
        "response": "Fine-tuning is the process of taking a pre-trained model and training it further on a specific dataset so it performs better for a particular task or domain."
    },
    {
        "instruction": "Why is full fine-tuning expensive?",
        "response": "Full fine-tuning updates all model parameters, so it requires more GPU memory, more training time, more storage, and higher cost."
    },
    {
        "instruction": "What is LoRA?",
        "response": "LoRA, or Low-Rank Adaptation, is a parameter-efficient fine-tuning method that freezes the base model and trains only small adapter matrices."
    },
    {
        "instruction": "What is QLoRA?",
        "response": "QLoRA is Quantized LoRA. It loads the base model in 4-bit precision and trains LoRA adapters, reducing memory usage while keeping good performance."
    },
    {
        "instruction": "What is the main difference between LoRA and QLoRA?",
        "response": "LoRA trains small adapter matrices on a normal precision base model, while QLoRA uses a 4-bit quantized base model to save more memory."
    },
    {
        "instruction": "What is the rank r in LoRA?",
        "response": "The rank r controls the size of the low-rank adapter matrices. A higher rank gives more capacity but uses more memory and compute."
    },
    {
        "instruction": "What is LoRA alpha?",
        "response": "LoRA alpha is a scaling factor that controls the strength of the LoRA update added to the frozen base model."
    },
    {
        "instruction": "What is LoRA dropout?",
        "response": "LoRA dropout randomly disables some adapter activations during training to reduce overfitting."
    },
    {
        "instruction": "When should we use LoRA?",
        "response": "Use LoRA when you want to adapt a model efficiently without updating all base model parameters."
    },
    {
        "instruction": "When should we use QLoRA?",
        "response": "Use QLoRA when GPU memory is limited and you want to fine-tune a larger model using 4-bit quantization with LoRA adapters."
    },
    {
        "instruction": "What is PEFT?",
        "response": "PEFT stands for Parameter-Efficient Fine-Tuning. It means adapting a model by training only a small number of parameters instead of the full model."
    },
    {
        "instruction": "What are adapters in LoRA?",
        "response": "Adapters are small trainable components added to the model layers. In LoRA, they learn task-specific updates while the base model remains frozen."
    },
    {
        "instruction": "What is an instruction dataset?",
        "response": "An instruction dataset contains prompts or questions along with ideal responses. It teaches the model how to respond to specific instructions."
    },
    {
        "instruction": "Give one use case of fine-tuning.",
        "response": "A banking chatbot can be fine-tuned on banking FAQs, policies, and customer support examples to give more domain-specific responses."
    },
    {
        "instruction": "What is the benefit of saving only LoRA adapters?",
        "response": "LoRA adapters are small, so they are easier to store, share, and load compared with saving the full fine-tuned model."
    },
    {
        "instruction": "What is quantization?",
        "response": "Quantization reduces the precision of model weights, such as from 16-bit to 4-bit, to reduce memory usage and make the model easier to run."
    },
    {
        "instruction": "Why do we evaluate after fine-tuning?",
        "response": "Evaluation checks whether the fine-tuned model gives better, safer, and more accurate answers on the target task."
    },
    {
        "instruction": "What is a common mistake in fine-tuning?",
        "response": "A common mistake is using low-quality or irrelevant data. Fine-tuning quality depends heavily on the quality of the dataset."
    },
]

df = pd.DataFrame(training_data)
df.head()

,instruction,response
0,What is Generative AI?,Generative AI is a type of artificial intellig...
1,Explain RAG in simple words.,RAG stands for Retrieval-Augmented Generation....
2,What is fine-tuning?,Fine-tuning is the process of taking a pre-tra...
3,Why is full fine-tuning expensive?,"Full fine-tuning updates all model parameters,..."
4,What is LoRA?,"LoRA, or Low-Rank Adaptation, is a parameter-e..."


,instruction,response
0,What is Generative AI?,Generative AI is a type of artificial intellig...
1,Explain RAG in simple words.,RAG stands for Retrieval-Augmented Generation....
2,What is fine-tuning?,Fine-tuning is the process of taking a pre-tra...
3,Why is full fine-tuning expensive?,"Full fine-tuning updates all model parameters,..."
4,What is LoRA?,"LoRA, or Low-Rank Adaptation, is a parameter-e..."


In [24]:
len(df)

20

20

# 9. Convert DataFrame to Hugging Face Dataset

In [25]:
dataset = Dataset.from_pandas(df)

dataset = dataset.train_test_split(
    test_size=0.15,
    seed=42
)

train_dataset = dataset["train"]
eval_dataset = dataset["test"]

print(train_dataset)
print(eval_dataset)

Dataset({
    features: ['instruction', 'response'],
    num_rows: 17
})
Dataset({
    features: ['instruction', 'response'],
    num_rows: 3
})
Dataset({
    features: ['instruction', 'response'],
    num_rows: 17
})
Dataset({
    features: ['instruction', 'response'],
    num_rows: 3
})


# 10. Load Tokenizer

The tokenizer converts text into tokens.

For chat models, we format examples as:

```text
system message
user instruction
assistant response
```


In [26]:
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Tokenizer loaded.")
print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer loaded.
Pad token: <|endoftext|>
EOS token: <|im_end|>
Tokenizer loaded.
Pad token: <|endoftext|>
EOS token: <|im_end|>


# 11. Format Dataset as Chat Examples

We convert each training row into a chat-style text format.

This helps the model learn how to respond as an assistant.


In [27]:
SYSTEM_MESSAGE = (
    "You are a helpful AI teacher. "
    "Answer clearly and simply for intermediate students."
)


def format_chat_example(example):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_MESSAGE
        },
        {
            "role": "user",
            "content": example["instruction"]
        },
        {
            "role": "assistant",
            "content": example["response"]
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {
        "text": text
    }


formatted_train = train_dataset.map(format_chat_example)
formatted_eval = eval_dataset.map(format_chat_example)

print(formatted_train[0]["text"])

Map:   0%|          | 0/17 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

<|im_start|>system
You are a helpful AI teacher. Answer clearly and simply for intermediate students.<|im_end|>
<|im_start|>user
What is the rank r in LoRA?<|im_end|>
<|im_start|>assistant
The rank r controls the size of the low-rank adapter matrices. A higher rank gives more capacity but uses more memory and compute.<|im_end|>



Map:   0%|          | 0/17 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

<|im_start|>system
You are a helpful AI teacher. Answer clearly and simply for intermediate students.<|im_end|>
<|im_start|>user
What is the rank r in LoRA?<|im_end|>
<|im_start|>assistant
The rank r controls the size of the low-rank adapter matrices. A higher rank gives more capacity but uses more memory and compute.<|im_end|>



# 12. Tokenize Dataset

We convert text into token IDs.

For this teaching demo, we train the model to predict the full formatted text.  
In advanced production training, you may mask the prompt tokens and train only on assistant responses.


In [28]:
def tokenize_function(example):
    tokenized = tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )

    tokenized["labels"] = tokenized["input_ids"].copy()

    return tokenized


tokenized_train = formatted_train.map(
    tokenize_function,
    batched=False,
    remove_columns=formatted_train.column_names
)

tokenized_eval = formatted_eval.map(
    tokenize_function,
    batched=False,
    remove_columns=formatted_eval.column_names
)

print(tokenized_train)
print(tokenized_eval)

Map:   0%|          | 0/17 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 17
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 3
})


Map:   0%|          | 0/17 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 17
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 3
})


# 13. Load Base Model

## For QLoRA

The model is loaded in **4-bit** using `BitsAndBytesConfig`.

## For LoRA

The model is loaded normally, and LoRA adapters are added.

> QLoRA requires GPU support. If you face issues, set `USE_QLORA = False` and restart runtime.


In [29]:
if USE_QLORA:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )

    model = prepare_model_for_kbit_training(model)

else:
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
        trust_remote_code=True,
    )

model.config.use_cache = False

print("Base model loaded.")

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Base model loaded.


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Base model loaded.


# 14. Add LoRA Adapters

For Qwen-style models, common target modules are:

```text
q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj
```

These are attention and feed-forward projection layers where LoRA can adapt the model efficiently.


In [30]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

print("LoRA adapters added.")
model.print_trainable_parameters()

LoRA adapters added.
trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497
LoRA adapters added.
trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


# 15. Training Arguments

Important hyperparameters:

| Hyperparameter | Meaning |
|---|---|
| `r` | LoRA rank / adapter size |
| `lora_alpha` | Scaling factor |
| `lora_dropout` | Dropout for adapter training |
| `learning_rate` | Step size during training |
| `batch_size` | Number of samples per step |
| `gradient_accumulation_steps` | Simulates larger batch size |
| `num_train_epochs` | Number of passes over dataset |


In [31]:
use_fp16 = torch.cuda.is_available()

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=1,
    eval_strategy="steps",
    eval_steps=5,
    save_steps=10,
    save_total_limit=2,
    fp16=use_fp16,
    report_to="none",
    optim="paged_adamw_8bit" if USE_QLORA else "adamw_torch",
)

print("Training arguments ready.")

Training arguments ready.
Training arguments ready.


# 16. Create Trainer

In [32]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
)

print("Trainer is ready.")

Trainer is ready.
Trainer is ready.


# 17. Start Fine-Tuning

This is the actual adapter training step.

For classroom demo, this may take a few minutes depending on GPU.


In [33]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
5,3.307041,3.176159


TrainOutput(global_step=5, training_loss=4.442342472076416, metrics={'train_runtime': 19.3405, 'train_samples_per_second': 0.879, 'train_steps_per_second': 0.259, 'total_flos': 19150348615680.0, 'train_loss': 4.442342472076416, 'epoch': 1.0})

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
5,3.306343,3.172769


TrainOutput(global_step=5, training_loss=4.44220290184021, metrics={'train_runtime': 24.1473, 'train_samples_per_second': 0.704, 'train_steps_per_second': 0.207, 'total_flos': 19150348615680.0, 'train_loss': 4.44220290184021, 'epoch': 1.0})

# 18. Save LoRA / QLoRA Adapter

Only adapter weights are saved.

This is much smaller than saving the full model.


In [34]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Adapter saved at:", OUTPUT_DIR)

Adapter saved at: /content/lora_qlora_adapter
Adapter saved at: /content/lora_qlora_adapter


# 19. Test the Fine-Tuned Model

Now we ask the model questions.

The model should answer in the style of our small instruction dataset.


In [35]:
def generate_answer(question, max_new_tokens=180):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_MESSAGE
        },
        {
            "role": "user",
            "content": question
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    model.eval()

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return response.strip()


print(generate_answer("Explain QLoRA in simple words."))

QLoRA stands for "Quantized Long Range", which is an approach to improve the quality of training deep learning models by using quantization, where each element of the model's data is represented with fewer bits than the original.

Think of it like adding more precision to your calculator when you're trying to solve complex math problems. You start with lots of decimal points (more digits) but as you get closer to solving the equation, you use only a few significant figures (fewer decimals).

In machine learning, QLoRA helps keep the model from getting stuck in small errors that can happen during training, making it easier to find better solutions and improving overall performance. It also reduces computation costs by reducing the number of parameters needed, leading to faster processing times.
QLoRA is short for Quantum LoRA, which stands for "Quantum Learning Regularization". It's like a special way of training machine learning models with quantum computers!


# 20. Try More Questions

In [38]:
questions = [
    "What is MBBS",
    "What is the difference between LoRA and QLoRA?",
    "Why is full fine-tuning expensive?",
    "What is the role of rank r in LoRA?",
    "When should we use QLoRA?"
]

for question in questions:
    print("=" * 80)
    print("Question:", question)
    print("Answer:", generate_answer(question))
    print()

Question: What is MBBS
Answer: MBBS stands for Medical Biology Bachelor of Medicine (Biology) Degree, also known as Medical Biology Bachelor of Science (MBS). It is a degree program in the medical field that prepares students to become doctors. This course requires significant academic work including courses in biology, genetics, and other relevant subjects. The MBBS program aims to provide students with foundational knowledge in medicine while offering an opportunity to specialize in various areas such as pediatrics, dermatology, or infectious diseases.

Question: What is the difference between LoRA and QLoRA?
Answer: LoRA stands for Learned Random Walk, which means it learns from the input data rather than being directly trained on that data. It's designed to be faster and more efficient than LLaMA.

QLoRA stands for Quantized Long Range, which is an improved version of LoRA. It uses quantization techniques to reduce the size of the weights, making training faster and using less memo

# 21. Load Adapter Later

After training, you can reload the adapter later.

This is useful when you want to share only adapter weights with students.


In [37]:
# Example reload code.
# Run this in a fresh runtime after training if needed.

'''
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
ADAPTER_PATH = "/content/lora_qlora_adapter"

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
'''
print("Reload example is available in this cell.")

Reload example is available in this cell.
Reload example is available in this cell.


# 22. LoRA vs QLoRA Comparison

| Feature | LoRA | QLoRA |
|---|---|---|
| Base model precision | Usually 16-bit / 32-bit | 4-bit quantized |
| Trainable weights | LoRA adapters only | LoRA adapters only |
| Memory usage | Lower than full fine-tuning | Much lower |
| Training cost | Lower | Very low |
| Best for | Small to medium models | Larger models with limited GPU memory |
| Main trade-off | Needs more memory than QLoRA | Slight quality or speed trade-off possible |


# 23. Common Problems and Fixes

## Problem 1: CUDA out of memory

Fixes:

- Use smaller model
- Reduce `MAX_LENGTH`
- Reduce batch size
- Use QLoRA instead of LoRA
- Restart runtime

## Problem 2: bitsandbytes error

Fixes:

- Make sure Colab runtime has GPU enabled
- Restart runtime after installing packages
- Set `USE_QLORA = False` if only CPU is available

## Problem 3: Poor model response after training

Reasons:

- Dataset is too small
- Data quality is poor
- Training epochs are too few
- Prompt format is inconsistent

## Problem 4: Overfitting

Reasons:

- Dataset is too small
- Too many epochs
- Learning rate too high


# 24. Student Lab Task

## Scenario

You are building a small AI teaching assistant for a Generative AI course.

The assistant should answer questions about:

- Fine-tuning
- LoRA
- QLoRA
- RAG
- Prompt engineering
- Embeddings

## Required Tasks

1. Run this notebook in Google Colab.
2. Explain the difference between fine-tuning and prompting.
3. Train LoRA or QLoRA adapters using the given dataset.
4. Add 10 more examples to the dataset.
5. Run training again.
6. Ask 5 test questions.
7. Compare responses before and after adding more data.
8. Explain why QLoRA saves GPU memory.
9. Take screenshots of training output and generated answers.
10. Submit your short reflection.


# 25. Extension Task

Advanced students can improve this notebook by:

1. Using a larger domain-specific dataset.
2. Loading data from CSV.
3. Masking prompt tokens and training only on assistant responses.
4. Evaluating with ROUGE, BLEU, or human scoring.
5. Uploading adapter weights to Hugging Face Hub.
6. Testing a larger model if GPU memory allows.


# Final Summary

Fine-tuning adapts a model to a specific task or domain.

LoRA makes fine-tuning efficient by training small adapter matrices instead of the full model.

QLoRA makes it even more memory-efficient by loading the base model in 4-bit precision while training LoRA adapters.

This notebook demonstrates the practical workflow:

```text
Prepare dataset
Load tokenizer
Load base model
Apply LoRA / QLoRA
Train adapters
Save adapters
Run inference
Evaluate results
```
